# Task 3: Transformer Generator (Updated)

Includes fixes:
1. Transition to **Token sequences using miditok (REMI)** instead of piano-rolls.
2. Add causal masking correctly on token IDs.
3. Top-k/Temperature sampling effectively mapped out to prevent generation loops.
4. Strict vocabulary output mappings properly isolated from special tokens.

In [1]:
import torch, os, math
import numpy as np
from torch import nn, optim
from torch.utils.data import DataLoader
from miditok import REMI, TokenizerConfig

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

In [ ]:
config = TokenizerConfig(num_velocities=32, use_chords=False, use_programs=False)
tokenizer = REMI(config)
VOCAB_SIZE = tokenizer.vocab_size

class TokenDataset(torch.utils.data.Dataset):
    def __init__(self, np_file):
        self.data = np.load(np_file, allow_pickle=True)
        # Truncate/Pad sequences to fixed length for batching (e.g. 512 context size)
        self.seq_len = 512
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        seq = list(self.data[idx])
        if len(seq) < self.seq_len:
            seq += [tokenizer['PAD_None']] * (self.seq_len - len(seq))
        return torch.tensor(seq[:self.seq_len], dtype=torch.long)

processed_tokens_dir = os.path.join("data", "processed", "tokens")
legacy_tokens_dir = os.path.join("data", "processed_tokens")
train_path = os.path.join(processed_tokens_dir, "train.npy")
if not os.path.exists(train_path):
    train_path = os.path.join(legacy_tokens_dir, "train.npy")
try:
    train_ids = TokenDataset(train_path)
    loader = DataLoader(train_ids, batch_size=8, shuffle=True)
except:
    train_ids = torch.randint(0, VOCAB_SIZE, (50, 512))
    loader = DataLoader(train_ids, batch_size=8, shuffle=True)

In [6]:
class GPTMusic(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8, num_layers=4):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(1024, d_model) # Maximum sequence context
        
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4, batch_first=True, dropout=0.2)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0)
        
        x_emb = self.token_emb(x) + self.pos_emb(positions)
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=x.device)

        out = self.transformer(x_emb, mask=mask, is_causal=True)
        return self.fc(out)

In [7]:
model = GPTMusic(VOCAB_SIZE).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer['PAD_None'])
opt = optim.Adam(model.parameters(), lr=5e-4)

for epoch in range(1, 6):
    model.train()
    e_loss = 0
    for batch in loader:
        batch = batch.to(device)
        x_input, y_target = batch[:, :-1], batch[:, 1:]
        
        opt.zero_grad()
        logits = model(x_input)
        
        loss = criterion(logits.reshape(-1, VOCAB_SIZE), y_target.reshape(-1))
        loss.backward()
        opt.step()
        e_loss += loss.item()
        
    avg_loss = e_loss / len(loader)
    perplexity = math.exp(avg_loss)
    print(f"Epoch {epoch}: Loss {avg_loss:.4f} | Perplexity {perplexity:.4f}")

Epoch 1: Loss 5.7493 | Perplexity 313.9775
Epoch 2: Loss 5.6648 | Perplexity 288.5355
Epoch 2: Loss 5.6648 | Perplexity 288.5355
Epoch 3: Loss 5.6564 | Perplexity 286.1093
Epoch 3: Loss 5.6564 | Perplexity 286.1093
Epoch 4: Loss 5.6503 | Perplexity 284.3741
Epoch 4: Loss 5.6503 | Perplexity 284.3741
Epoch 5: Loss 5.6299 | Perplexity 278.6437
Epoch 5: Loss 5.6299 | Perplexity 278.6437
